## Enable GPU

In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
!nvidia-smi

Sat Mar 21 13:02:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Imports + Config

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -----------------------
# Config
# -----------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 16          # ⬆ Increased from 8 (better gradient stability)
EPOCHS = 30

TRAIN_PATH = r"C:\Users\tanis\Downloads\Cat_emotion_classification_Project\data_analysis\datasets\imagefiles\IMAGES_Dataset\train"
VAL_PATH = r"C:\Users\tanis\Downloads\Cat_emotion_classification_Project\data_analysis\datasets\imagefiles\IMAGES_Dataset\val"


In [11]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [4]:
# -----------------------
# Load Train Dataset
# -----------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# -----------------------
# Load Validation Dataset
# -----------------------
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -----------------------
# Get class info (FIXED missing num_classes)
# -----------------------
class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

# -----------------------
# Performance optimization
# -----------------------
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# -----------------------
# Compute Class Weights (NEW - Handles class imbalance)
# -----------------------
labels = []

for _, y in train_ds:
    labels.extend(y.numpy())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weight_dict = dict(enumerate(class_weights))

print("Class Weights:", class_weight_dict)

# -----------------------
# Data Augmentation (NEW - improves generalization)
# -----------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\tanis\\Downloads\\Cat_emotion_classification_Project\\data_analysis\\datasets\\imagefiles\\IMAGES_Dataset\\train'

## Dataset Check (YOU ASKED ABOUT THIS)

In [ ]:
for x, y in train_ds.take(1):
    print(x.shape, y.shape)

## Class Info + Prefetch

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## Student Model(Resnet50)

In [ ]:
# -----------------------
# ResNet50 Base Model
# -----------------------
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# -----------------------
# Fine-Tuning Strategy (IMPROVED)
# Freeze early layers, train last 30
# -----------------------
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

# -----------------------
# Build Classifier Head (IMPROVED)
# -----------------------
inputs = keras.Input(shape=(224, 224, 3))

# 🔥 Apply augmentation BEFORE preprocessing
x = data_augmentation(inputs)

# Preprocess for ResNet
x = preprocess_input(x)

# Forward pass through base model
x = base_model(x, training=True)  # training=True for fine-tuning

# Global pooling
x = layers.GlobalAveragePooling2D()(x)

# 🔥 Stronger classifier head
x = layers.Dense(128, activation="relu")(x)
x = layers.BatchNormalization()(x)   # NEW - stabilizes learning
x = layers.Dropout(0.5)(x)           # Increased dropout

outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)


In [ ]:
teacher_model = EfficientNetB4(
    weights = "imagenet",
    include_top = True
)

## Distillation Loss

In [ ]:
import tensorflow as tf
import numpy as np

class DistillationLoss:
    def __init__(self, temperature=4.0, alpha=0.5):
        self.temperature = temperature
        self.alpha = alpha
        self.ce = tf.keras.losses.SparseCategoricalCrossentropy()
        self.kld = tf.keras.losses.KLDivergence()

    def __call__(self, y_true, student_logits, teacher_logits):
        T = self.temperature

        hard_loss = self.ce(y_true, student_logits)

        soft_student = tf.nn.softmax(student_logits / T)
        soft_teacher = tf.nn.softmax(teacher_logits / T)

        soft_loss = self.kld(soft_teacher, soft_student) * (T ** 2)

        return self.alpha * hard_loss + (1 - self.alpha) * soft_loss

## Temperature Scheduler

In [ ]:
class TemperatureScheduler:
    def __init__(self, initial_temp=5.0, final_temp=1.0, total_epochs=EPOCHS):
        self.initial_temp = initial_temp
        self.final_temp = final_temp
        self.total_epochs = total_epochs

    def get_temperature(self, epoch):
        progress = epoch / (self.total_epochs - 1)
        return self.initial_temp + (self.final_temp - self.initial_temp) * progress

## Teacher Function

In [ ]:
def generate_teacher_logits(labels, num_classes):
    batch_size = tf.shape(labels)[0]

    logits = tf.random.normal((batch_size, num_classes))

    indices = tf.stack([tf.range(batch_size), labels], axis=1)
    updates = tf.ones(batch_size) * 3.0

    logits = tf.tensor_scatter_nd_add(logits, indices, updates)

    return logits

Optimizer

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)

Train Step

In [ ]:
@tf.function
def train_step(images, labels, model, loss_fn, temperature):
    teacher_logits = generate_teacher_logits(labels, num_classes)

    with tf.GradientTape() as tape:
        student_logits = model(images, training=True)

        loss_fn.temperature = temperature
        loss = loss_fn(labels, student_logits, teacher_logits)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return loss

Validation Step

In [ ]:
@tf.function
def val_step(images, labels, model):
    preds = model(images, training=False)

    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, preds)
    acc = tf.keras.metrics.sparse_categorical_accuracy(labels, preds)

    return tf.reduce_mean(loss), tf.reduce_mean(acc)

FINAL TRAINING LOOP

In [ ]:
loss_fn = DistillationLoss()
temp_scheduler = TemperatureScheduler()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    temperature = temp_scheduler.get_temperature(epoch)

    # Training
    for images, labels in train_ds:
        loss = train_step(images, labels, model, loss_fn, temperature)

    # Validation
    val_losses = []
    val_accs = []

    for images, labels in val_ds:
        v_loss, v_acc = val_step(images, labels, model)
        val_losses.append(v_loss.numpy())
        val_accs.append(v_acc.numpy())

    print(f"Loss: {loss.numpy():.4f}")
    print(f"Val Loss: {np.mean(val_losses):.4f}")
    print(f"Val Acc: {np.mean(val_accs):.4f}")

In [ ]:
model.save("distilled_resnet50.keras")